# 01 - Extraer embeddings con ResNet18

Flujo:
1. Cargar manifests train/test.
2. Split train/val por `patient_id`.
3. Cargar `resnet18_tuned_best.pt` como backbone congelado.
4. Extraer embeddings 512-D y normalizar con z-score (stats de train).
5. Guardar en `Prueba_emmbedings/artifacts/embeddings.pt`.

In [1]:
# Imports y configuración
from pathlib import Path
import sys

import torch
from torch.utils.data import DataLoader

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "Prueba_emmbedings" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "scripts"))

import notebook_utils as nu
from paths import MANIFEST_TRAIN, MANIFEST_TEST, REPORTS_MODELS_DIR

SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 16
ARTIFACTS_DIR = ROOT / "Prueba_emmbedings" / "artifacts"
BACKBONE_PATH = REPORTS_MODELS_DIR / "resnet18_tuned_best.pt"
OUT_EMB = ARTIFACTS_DIR / "embeddings.pt"

nu.seed_everything(SEED)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

print("Backbone:", BACKBONE_PATH)
print("Manifests:", MANIFEST_TRAIN, MANIFEST_TEST)

Backbone: G:\Cosas_programacion\Breast Cancer Interpretable-ml\reports\models\resnet18_tuned_best.pt
Manifests: G:\Cosas_programacion\Breast Cancer Interpretable-ml\src\data\processed\manifest_train.csv G:\Cosas_programacion\Breast Cancer Interpretable-ml\src\data\processed\manifest_test.csv


In [2]:
# Cargar datos y DataLoaders
tr_df, val_df, test_df = nu.load_manifest_splits(MANIFEST_TRAIN, MANIFEST_TEST, seed=SEED, test_size=0.2)

eval_tfms = nu.default_transforms(img_size=IMG_SIZE, augment=False)
train_ds = nu.MammographyDataset(tr_df, transform=eval_tfms)
val_ds = nu.MammographyDataset(val_df, transform=eval_tfms)
test_ds = nu.MammographyDataset(test_df, transform=eval_tfms)

loaders = {
    "train": DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0),
    "val": DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0),
    "test": DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0),
}

print("Train:", len(train_ds), "Val:", len(val_ds), "Test:", len(test_ds))

Train: 2315 Val: 549 Test: 422


In [3]:
# Extraer embeddings y guardar artifact
device = nu.pick_device(prefer_cpu=False)
backbone = nu.build_resnet18_backbone(BACKBONE_PATH, device=device)
emb, (mu, sigma) = nu.extract_and_normalize_embeddings(backbone, loaders, device=device)

Xtr, ytr = emb["train"]
Xva, yva = emb["val"]
Xte, yte = emb["test"]

payload = {
    "seed": SEED,
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "backbone_path": str(BACKBONE_PATH.as_posix()),
    "X_train": Xtr,
    "y_train": ytr,
    "X_val": Xva,
    "y_val": yva,
    "X_test": Xte,
    "y_test": yte,
    "mu": mu,
    "sigma": sigma,
}

torch.save(payload, OUT_EMB)
print("Embeddings guardados en:", OUT_EMB)
print("Shapes -> train:", tuple(Xtr.shape), "val:", tuple(Xva.shape), "test:", tuple(Xte.shape))

Embeddings guardados en: G:\Cosas_programacion\Breast Cancer Interpretable-ml\Prueba_emmbedings\artifacts\embeddings.pt
Shapes -> train: (2315, 512) val: (549, 512) test: (422, 512)
